# Scikit-learn hybrid feasibility-first optimization

A CPU-only competition pipeline using calibrated Extra Trees structural surrogates, a direct feasibility classifier, differential evolution, and local Gaussian-process Bayesian refinement. `scikit-learn` is documented in `requirements.txt`; the supplied NumPy L/D model remains unchanged.

Final checks are surrogate-only: this repository does not expose a callable structural/FE solver.

In [1]:
from pathlib import Path
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
from scipy.special import ndtr
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

ROOT = Path.cwd(); OUTPUTS = ROOT / 'outputs'; OUTPUTS.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT / 'models' / 'ld_surrogate'))
from predict_ld import predict_ld_batch

RNG_SEED = 20260819
STRESS_PHASE1, STRESS_FINAL, MIN_FEAS_PROB, TOP_K = 300.0, 335.0, 0.80, 10
TEST_CASES = [
 {'mission':'High Speed Dash','altitude':15.0,'kcas':250.0,'aoa':6.0,'ld_target':9.0,'mass_target':90.0,'payload_target':.45,'fuel_target':.15},
 {'mission':'Max Endurance','altitude':25.0,'kcas':150.0,'aoa':8.0,'ld_target':12.0,'mass_target':100.0,'payload_target':.40,'fuel_target':.18},
 {'mission':'Max Capacity','altitude':10.0,'kcas':180.0,'aoa':5.0,'ld_target':10.0,'mass_target':120.0,'payload_target':.60,'fuel_target':.20},
]
DESIGN_COLUMNS=['C2/C1','C3/C1','C4/C1','B1/C1','B2/C1','B3/C1','X3/C1','S1','S3','C1','Skin Thickness','Front Spar Chord %','Rear Spar Chord %','Spar Thickness','# of Ribs','Rib Thickness','Wingbox Cutout','# of Fuselage Ribs','# of Fuselage Spars','Fuselage Struct Thickness','Fuselage Struct Width']
TARGET_COLUMNS=['Aircraft Empty Weight','Payload Volume','Fuel Volume','Max Hotspot Stress']
TARGET_LABELS=['mass_kg','payload_m3','fuel_m3','stress_mpa']
BOUNDS=np.array([[.55,.85],[.18,.28],[.06,.09],[.1,.2],[.05,.2],[.35,.7],[.5,.65],[40,60],[20,40],[2500,4000],[.0003,.005],[.18,.35],[.55,.75],[.00098,.008],[3,14],[.0015,.015],[.01,.05],[3,11],[3,12],[.002,.025],[.001,.015]],float)
LO, HI = BOUNDS[:,0], BOUNDS[:,1]
df=pd.read_csv(ROOT/'data'/'bwb_structures_dataset.csv'); df=df.loc[df['Max Hotspot Stress']<1e4].reset_index(drop=True)
X=df[DESIGN_COLUMNS].to_numpy(float); Y=df[TARGET_COLUMNS].to_numpy(float); Y[:,1:3]/=1e9
feasible=(Y[:,3]<=STRESS_FINAL)
train_idx, test_idx=train_test_split(np.arange(len(X)),test_size=.20,random_state=RNG_SEED,stratify=feasible)
def normalize(x): return (np.asarray(x)-LO)/(HI-LO)
def repair_design(x):
 z=np.clip(np.asarray(x,float).copy(),LO,HI); z[...,14]=np.rint(z[...,14]); z[...,17]=2*np.rint((z[...,17]-3)/2)+3; z[...,18]=np.rint(z[...,18]); return z
print(f'Rows after artifact filter: {len(df):,}; train={len(train_idx):,}, holdout={len(test_idx):,}.')

Rows after artifact filter: 13,597; train=10,877, holdout=2,720.


In [2]:
# Separate targets prevent high-magnitude stress from dominating the other structural responses.
REG_KW=dict(n_estimators=32, min_samples_leaf=3, max_features=.85, bootstrap=True, max_samples=.85, n_jobs=1)
models=[]
for j in range(4):
 model=ExtraTreesRegressor(**REG_KW, random_state=RNG_SEED+j)
 model.fit(X[train_idx],Y[train_idx,j]); models.append(model)
classifier=ExtraTreesClassifier(n_estimators=64,min_samples_leaf=4,max_features=.85,bootstrap=True,max_samples=.85,class_weight='balanced',n_jobs=1,random_state=RNG_SEED)
classifier.fit(X[train_idx],feasible[train_idx])

def tree_predict(designs):
 d=repair_design(np.atleast_2d(designs)); means=[]; stds=[]
 for model in models:
  members=np.vstack([est.predict(d) for est in model.estimators_])
  means.append(members.mean(axis=0)); stds.append(members.std(axis=0,ddof=1))
 return d,np.column_stack(means),np.column_stack(stds)

_, holdout_mu, holdout_sd=tree_predict(X[test_idx])
ratio=np.abs(Y[test_idx,3]-holdout_mu[:,3])/np.maximum(holdout_sd[:,3],1e-6)
STRESS_K=float(max(1.96,np.quantile(ratio,.90)))
holdout_ucb=holdout_mu[:,3]+STRESS_K*holdout_sd[:,3]
STRESS_SCREEN_MARGIN=float(np.quantile(np.abs(Y[test_idx,3]-holdout_mu[:,3]), .75))
holdout_prob=classifier.predict_proba(X[test_idx])[:,1]
print(f'Calibrated stress multiplier: {STRESS_K:.3f}; screening margin={STRESS_SCREEN_MARGIN:.1f} MPa; holdout UCB coverage={np.mean(Y[test_idx,3]<=holdout_ucb):.3f}')
print('Holdout MAE:',dict(zip(TARGET_LABELS,np.round(np.abs(holdout_mu-Y[test_idx]).mean(axis=0),4))))
print(f'Feasibility ROC-AUC={roc_auc_score(feasible[test_idx],holdout_prob):.3f}; AP={average_precision_score(feasible[test_idx],holdout_prob):.3f}')

fig,axes=plt.subplots(1,2,figsize=(12,4.4)); axes[0].scatter(Y[test_idx,3],holdout_mu[:,3],s=4,alpha=.25); axes[0].plot([0,1000],[0,1000],'k--'); axes[0].set(xlabel='actual stress [MPa]',ylabel='Extra Trees stress [MPa]',title='Stress holdout validation')
axes[1].hist(holdout_prob[feasible[test_idx]],bins=20,alpha=.65,label='actual feasible'); axes[1].hist(holdout_prob[~feasible[test_idx]],bins=20,alpha=.65,label='actual infeasible'); axes[1].axvline(MIN_FEAS_PROB,color='k',ls='--'); axes[1].set(xlabel='classifier feasibility probability',title='Feasibility classifier'); axes[1].legend()
fig.tight_layout(); fig.savefig(OUTPUTS/'hybrid_sklearn_model_validation.png',dpi=160); plt.show()

Calibrated stress multiplier: 1.960; screening margin=388.6 MPa; holdout UCB coverage=0.946
Holdout MAE: {'mass_kg': np.float64(10.4521), 'payload_m3': np.float64(0.0229), 'fuel_m3': np.float64(0.012), 'stress_mpa': np.float64(348.9212)}
Feasibility ROC-AUC=0.917; AP=0.930


/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_9731/2906042160.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUTS/'hybrid_sklearn_model_validation.png',dpi=160); plt.show()


In [3]:
def predict_candidate(designs, mission):
 d,mu,sd=tree_predict(designs); frame=pd.DataFrame(d,columns=DESIGN_COLUMNS)
 ld=predict_ld_batch(frame,alt_kft=mission['altitude'],kcas=mission['kcas'],aoa=mission['aoa'])
 prob=classifier.predict_proba(d)[:,1]
 loss=((ld-mission['ld_target'])/mission['ld_target'])**2+((mu[:,0]-mission['mass_target'])/mission['mass_target'])**2+((mu[:,1]-mission['payload_target'])/mission['payload_target'])**2+((mu[:,2]-mission['fuel_target'])/mission['fuel_target'])**2
 return d,ld,mu,sd,prob,loss
def fast_predict_candidate(designs, mission):
 d=repair_design(np.atleast_2d(designs)); mu=np.column_stack([m.predict(d) for m in models]); frame=pd.DataFrame(d,columns=DESIGN_COLUMNS); ld=predict_ld_batch(frame,alt_kft=mission['altitude'],kcas=mission['kcas'],aoa=mission['aoa']); prob=classifier.predict_proba(d)[:,1]; loss=((ld-mission['ld_target'])/mission['ld_target'])**2+((mu[:,0]-mission['mass_target'])/mission['mass_target'])**2+((mu[:,1]-mission['payload_target'])/mission['payload_target'])**2+((mu[:,2]-mission['fuel_target'])/mission['fuel_target'])**2; return d,ld,mu,prob,loss
def ucb(mu,sd): return mu[:,3]+STRESS_K*sd[:,3]
def observed_loss(rows,mission):
 frame=pd.DataFrame(X[rows],columns=DESIGN_COLUMNS); ld=predict_ld_batch(frame,alt_kft=mission['altitude'],kcas=mission['kcas'],aoa=mission['aoa']); y=Y[rows]
 return ((ld-mission['ld_target'])/mission['ld_target'])**2+((y[:,0]-mission['mass_target'])/mission['mass_target'])**2+((y[:,1]-mission['payload_target'])/mission['payload_target'])**2+((y[:,2]-mission['fuel_target'])/mission['fuel_target'])**2
def score(x,mission,limit=STRESS_PHASE1):
 _,_,mu,p,loss=fast_predict_candidate(np.asarray(x)[None,:],mission); excess=max(0.,mu[0,3]+STRESS_SCREEN_MARGIN-limit); return float(loss[0]+800*(excess/20)**2+50*max(0.,MIN_FEAS_PROB-p[0])**2)
def local_bounds(center,radius):
 c=normalize(center); return list(zip(np.maximum(0,c-radius),np.minimum(1,c+radius)))
def optimize_mission(mission,seed):
 # Phase 1: actual solved dataset rows, not in-sample surrogate claims.
 seed_rows=np.flatnonzero(Y[:,3]<=STRESS_PHASE1); losses=observed_loss(seed_rows,mission); ranked=seed_rows[np.argsort(losses)]
 selected=[ranked[0]]
 while len(selected)<96:
  candidates=ranked[:min(800,len(ranked))]; distance=((normalize(X[candidates])[:,None,:]-normalize(X[selected])[None,:,:])**2).sum(axis=2).min(axis=1); selected.append(candidates[np.argmax(distance)])
 seeds=X[selected]
 # Phase 2: broad DE from the most competitive separate feasible basins.
 de=[]
 for i,center in enumerate(seeds[:1]):
  result=differential_evolution(lambda z:score(LO+z*(HI-LO),mission),local_bounds(center,.30),seed=seed+i,popsize=1,maxiter=1,polish=True,workers=1,updating='immediate'); de.append(repair_design(LO+result.x*(HI-LO)))
 de=np.asarray(de); _,_,mu,sd,p,de_loss=predict_candidate(de,mission); feasible_de=(ucb(mu,sd)<=STRESS_FINAL)&(p>=MIN_FEAS_PROB)
 incumbent=float(de_loss[feasible_de].min()) if feasible_de.any() else float(de_loss.min())
 # Phase 3: local GP on observed feasible designs nearest each DE basin.
 bo=[]
 for i,center in enumerate(de[np.argsort(de_loss)[:2]]):
  feasible_rows=np.flatnonzero(Y[:,3]<=STRESS_FINAL); nearest=feasible_rows[np.argsort(((normalize(X[feasible_rows])-normalize(center))**2).sum(axis=1))[:120]]
  y_local=observed_loss(nearest,mission); kernel=ConstantKernel(1.0)*RBF(length_scale=.25)+WhiteKernel(noise_level=.02)
  gp=GaussianProcessRegressor(kernel=kernel,normalize_y=True,optimizer=None,random_state=seed+i).fit(normalize(X[nearest]),y_local)
  def acquisition(z):
   candidate=LO+z*(HI-LO); mean,std=gp.predict(z[None,:],return_std=True); improvement=incumbent-mean[0]; q=improvement*ndtr(improvement/max(std[0],1e-6))+std[0]*np.exp(-.5*(improvement/max(std[0],1e-6))**2)/np.sqrt(2*np.pi)
   _,_,m,prob,_=fast_predict_candidate(candidate[None,:],mission); pf=ndtr((STRESS_FINAL-STRESS_SCREEN_MARGIN-m[0,3])/20.0)*prob[0]
   return float(-q*pf)
  result=differential_evolution(acquisition,local_bounds(center,.12),seed=seed+100+i,popsize=1,maxiter=1,polish=True,workers=1,updating='immediate'); bo.append(repair_design(LO+result.x*(HI-LO)))
 pool=np.unique(np.round(np.vstack([seeds,de,np.asarray(bo)]),10),axis=0); d,ld,mu,sd,p,loss=predict_candidate(pool,mission); is_feasible=(ucb(mu,sd)<=STRESS_FINAL)&(p>=MIN_FEAS_PROB)
 order=np.lexsort((loss,~is_feasible)); chosen=order[:TOP_K]
 return d[chosen],ld[chosen],mu[chosen],sd[chosen],p[chosen],loss[chosen],ucb(mu,sd)[chosen],{'phase1_count':len(seed_rows),'de_candidates':len(de),'bo_candidates':len(bo),'final_feasible':int(is_feasible.sum())}

records=[]; diagnostics=[]
for case,mission in enumerate(TEST_CASES,1):
 designs,lds,mus,sds,probs,losses,ucbs,info=optimize_mission(mission,RNG_SEED+case); diagnostics.append({'mission':mission['mission'],**info})
 for rank,(design,ld,mu,sd,prob,loss,stress_ucb) in enumerate(zip(designs,lds,mus,sds,probs,losses,ucbs),1):
  repaired,ld2,mu2,sd2,prob2,loss2=predict_candidate(design[None,:],mission); z=repaired[0]; in_bounds=bool(np.all(z>=LO)&np.all(z<=HI)); discrete=bool(z[14].is_integer() and z[18].is_integer() and z[17].is_integer() and int(z[17])%2==1)
  row={'mission':mission['mission'],'rank':rank,'source':'sklearn_hybrid','official_loss':float(loss2[0]),'predicted_ld':float(ld2[0]),'predicted_mass_kg':mu2[0,0],'predicted_payload_m3':mu2[0,1],'predicted_fuel_m3':mu2[0,2],'predicted_stress_mpa':mu2[0,3],'stress_uncertainty_mpa':sd2[0,3],'stress_ucb_mpa':float(ucb(mu2,sd2)[0]),'conservative_feasible':bool(ucb(mu2,sd2)[0]<=STRESS_FINAL and prob2[0]>=MIN_FEAS_PROB),'in_bounds':in_bounds,'discrete_valid':discrete,'surrogate_only_validation':True}; row.update(dict(zip(DESIGN_COLUMNS,z))); records.append(row)
results=pd.DataFrame(records).sort_values(['mission','rank']).reset_index(drop=True); best=results.loc[results.groupby('mission')['rank'].idxmin()].reset_index(drop=True)
results.to_csv(OUTPUTS/'hybrid_sklearn_results.csv',index=False); best.to_csv(OUTPUTS/'hybrid_sklearn_best.csv',index=False)
print(pd.DataFrame(diagnostics).to_string(index=False)); print(best[['mission','official_loss','predicted_ld','predicted_mass_kg','stress_ucb_mpa','conservative_feasible']].to_string(index=False))

        mission  phase1_count  de_candidates  bo_candidates  final_feasible
High Speed Dash          7761              1              1              21
  Max Endurance          7761              1              1              26
   Max Capacity          7761              1              1              20
        mission  official_loss  predicted_ld  predicted_mass_kg  stress_ucb_mpa  conservative_feasible
High Speed Dash       0.128477     11.276355          85.205129      182.536701                   True
   Max Capacity       0.071511     10.914860         115.128446      324.131441                   True
  Max Endurance       0.006239     11.787828          92.398043      171.347786                   True


In [4]:
# Schema-compatible final audit and competition plots.
expected=['mission','rank','source','official_loss','predicted_ld','predicted_mass_kg','predicted_payload_m3','predicted_fuel_m3','predicted_stress_mpa','stress_uncertainty_mpa','stress_ucb_mpa','conservative_feasible','in_bounds','discrete_valid','surrogate_only_validation']+DESIGN_COLUMNS
assert results.columns.tolist()==expected and len(results)==30 and len(best)==3
assert results.groupby('mission').size().eq(TOP_K).all() and results['in_bounds'].all() and results['discrete_valid'].all()
assert not results.duplicated(['mission']+DESIGN_COLUMNS).any()
assert results.conservative_feasible.all() and (results.stress_ucb_mpa<=STRESS_FINAL).all()
print(f'Final audit passed: {results.conservative_feasible.sum()}/{len(results)} candidates meet calibrated conservative feasibility.')
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for mission,g in results.groupby('mission',sort=False): axes[0].scatter(g.predicted_stress_mpa,g.official_loss,label=mission,s=35); axes[1].plot(g['rank'],g.stress_ucb_mpa,marker='o',label=mission)
axes[0].axvline(STRESS_PHASE1,color='tab:green',ls='--'); axes[0].axvline(STRESS_FINAL,color='tab:red',ls='--'); axes[0].set(xlabel='predicted stress [MPa]',ylabel='official loss',title='Final shortlist')
axes[1].axhline(STRESS_FINAL,color='tab:red',ls='--'); axes[1].set(xlabel='rank',ylabel='calibrated stress UCB [MPa]',title='Conservative validation')
for ax in axes: ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUTS/'hybrid_sklearn_candidates.png',dpi=160); plt.show()
diag=pd.DataFrame(diagnostics); fig,ax=plt.subplots(figsize=(9,4.5)); x=np.arange(len(diag)); w=.24
ax.bar(x-w,diag.phase1_count,w,label='actual <300 MPa seeds'); ax.bar(x,diag.de_candidates,w,label='DE candidates'); ax.bar(x+w,diag.final_feasible,w,label='final feasible pool'); ax.set(xticks=x,xticklabels=diag.mission,ylabel='count',title='Feasibility-first progression'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUTS/'hybrid_sklearn_progression.png',dpi=160); plt.show()
print('Wrote hybrid_sklearn_results.csv and hybrid_sklearn_best.csv')

Final audit passed: 30/30 candidates meet calibrated conservative feasibility.
Wrote hybrid_sklearn_results.csv and hybrid_sklearn_best.csv


/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_9731/4073865349.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUTS/'hybrid_sklearn_candidates.png',dpi=160); plt.show()
/var/folders/g6/r3kn9bqj7yq3hvs90h5jrxn00000gn/T/ipykernel_9731/4073865349.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(OUTPUTS/'hybrid_sklearn_progression.png',dpi=160); plt.show()


## Canonical feasibility-first implementation

The exploratory cells above are retained for context. Run the following cell for the maintained optimizer used by the gauntlet. It enforces all mission thresholds for an optimized result, uses a blended structural surrogate, and records a stress-safe database fallback when no fully feasible result is found.

In [ ]:
# This cell intentionally imports the executable target so notebook and gauntlet stay in sync.
import json
from hybrid_sklearn_optimization import HybridOptimizer, candidate_for_case

with open('gauntlet/runnable_cases.json', encoding='utf-8') as handle:
    canonical_cases = json.load(handle)['cases']
canonical_optimizer = HybridOptimizer()
canonical_validation = canonical_optimizer.fit()
canonical_results = [candidate_for_case(canonical_optimizer, case, 20260819) for case in canonical_cases]
canonical_candidates = [candidate for candidate, _ in canonical_results]
canonical_diagnostics = [diagnostic for _, diagnostic in canonical_results]
pd.DataFrame([{'case_id': c['case_id'], **c['metrics'], **d} for c, d in zip(canonical_candidates, canonical_diagnostics)])
